# Sudoku Knowledge Representation & Inference

You will implement three functions -- `build_general_kb`, `build_definite_kb`, `pl_bc_entails` -- and the full-grid solving logic in `sudoku_solver.py`.

This notebook imports and tests those functions. The Streamlit app (`sudoku_app.py`) must import the same implementation from `sudoku_solver.py`; do not copy or rewrite the solver functions inside the app.

**Rules:**

- In `sudoku_solver.py`, import only from `utils.py` and `logic_.py`; do not modify either file.
- Do not duplicate the core solver functions in this notebook or `sudoku_app.py`.
- All other content in this notebook may be edited freely.


In [1]:
from utils import *
from logic_ import *
import json
import time
import importlib
import sudoku_solver

# Reload so edits made to sudoku_solver.py are picked up when this cell is rerun.
importlib.reload(sudoku_solver)

from sudoku_solver import (
    atom,
    build_general_kb,
    build_definite_kb,
    solve_full_grid_fc,
    pl_bc_entails,
    solve_full_grid_bc,
)


## Loading a puzzle from JSON

Puzzles are provided as JSON, not embedded in this notebook. Each file looks like:

```json
{
  "n": 9, "box_h": 3, "box_w": 3,
  "puzzles": [
    {
      "givens": {"1_1": 3, "2_3": 1, ...},
      "given_count": 28,
      "solution": {"1_1": 3, "1_2": 4, ...}
    },
    ...
  ]
}
```

`"r_c"` string keys map to the value at row `r`, column `c` (1-indexed). `given_count` is exactly how many cells are given. `solution` is included so you can check your own work as you go, but your functions must not read `solution` to answer a query, they should only read `givens`.

In [2]:
#do not change this function, it is used to load the puzzle pool from a json file
def load_pool(path):
    with open(path) as f:
        raw = json.load(f)
    puzzles = []
    for p in raw['puzzles']:
        givens = {tuple(int(x) for x in k.split('_')): v for k, v in p['givens'].items()}
        solution = {tuple(int(x) for x in k.split('_')): v for k, v in p['solution'].items()}
        puzzles.append({'givens': givens, 'solution': solution, 'given_count': p['given_count']})
    return raw['n'], raw['box_h'], raw['box_w'], puzzles

n, box_h, box_w, puzzle_pool = load_pool('puzzles.json')
print(f'{len(puzzle_pool)} puzzles loaded, {n}x{n} grid, {box_h}x{box_w} boxes')
print('given_count values:', sorted(p['given_count'] for p in puzzle_pool))

# Pick one puzzle to work with through the rest of this notebook.
puzzle = puzzle_pool[0]
givens = puzzle['givens']
print(f"working puzzle has {puzzle['given_count']} givens")
for r in range(1, n + 1):
    print([givens.get((r, c), '.') for c in range(1, n + 1)])

5 puzzles loaded, 9x9 grid, 3x3 boxes
given_count values: [30, 33, 36, 39, 42]
working puzzle has 30 givens
['.', 3, '.', '.', '.', '.', '.', '.', '.']
[2, '.', '.', '.', '.', 7, 8, 6, '.']
[5, 8, '.', 2, 6, '.', '.', 3, '.']
[7, 5, '.', '.', '.', '.', '.', 8, '.']
['.', '.', '.', '.', 7, '.', 5, '.', 4]
['.', '.', '.', 5, 3, '.', '.', 9, 6]
['.', 1, 2, '.', '.', 9, '.', '.', '.']
[6, 4, '.', '.', 5, 8, 9, '.', '.']
['.', '.', '.', '.', 2, 3, '.', '.', '.']


Notice: `pl_fc_entails`/`pl_resolution`/`tt_entails` are all given to you, already implemented, in `logic_.py`. The one algorithm you write yourself in this assignment is **backward chaining** (`pl_bc_entails`) -- see Task 2.

## Define Symbols

Two families of propositional symbols, for row $r$, column $c$, value $v$ (all ranging over $1$ to $n$):

| Symbol | Meaning |
|---|---|
| $\mathit{Is}_{rcv}$ | cell $(r,c)$ has value $v$ |
| $\mathit{Not}_{rcv}$ | cell $(r,c)$ does **not** have value $v$ |

For each representation in Task 1, determine which of these two families is actually needed.

Hint: consider what form a definite (Horn) clause must take, and what `PropDefiniteKB.tell()` will accept.

### Helper function

`atom(prefix, r, c, v)`, where prefix can be either `Is` or `Not`, is a naming helper so propositional symbols need not be typed. It is provided for you in `sudoku_solver.py`; do not change it.

In code, $\mathit{Is}_{rcv}$ and $\mathit{Not}_{rcv}$ are written as single-word symbol names, e.g. `Is3_2_4` and `Not3_2_4`: the prefix is followed immediately by `r`, then by `c` and `v` separated by underscores. No separator is needed between the prefix and `r` since `expr()` only requires a symbol name to start with an uppercase letter. So `Is3_2_4` is parsed as one valid symbol.


In [ ]:
# atom() is provided in sudoku_solver.py and imported above.


## Part A: Design the Sudoku Solver

### A.1) Knowledge Representation - Build KB

Every well-posed Sudoku puzzle satisfies exactly these conditions:

- Each cell is assigned **at least one** value from $\{1, \dots, n\}$.
- Each cell is assigned **at most one** value from $\{1, \dots, n\}$, i.e., it cannot hold two different values at once.
- No two cells in the same row hold the same value.
- No two cells in the same column hold the same value.
- No two cells in the same box hold the same value.
- The **givens** cells hold their stated values.

Formalize these Sudoku constraints as propositional logic, in **two** representations:

**(a) General clauses -  `build_general_kb`.**

Encode each of the following directly: no restriction here; you may use arbitrary disjunctions of positive or negated literals. Must return a `PropKB`. 

**(b) Definite (Horn) clauses:- `build_definite_kb`.** Must return a `PropDefiniteKB`. Recall a definite clause is a disjunction of literals with exactly one *positive* literal.

Equivalently written as an implication whose conclusion is a single positive literal and whose premises are a conjunction of positive literals: `P1 & P2 & ... & Pk ==> Q`.

`PropDefiniteKB.tell()` will reject anything else.

Both functions take the puzzle's givens as fixed facts.

- **Implement `build_general_kb()` and `build_definite_kb()` in sudoku_solver.py**
- **Rerun the import cell near the top of this notebook after making changes.**
- **Explain your representation in Conceptual Question 1 below.**

In [ ]:
# Implement build_general_kb() and build_definite_kb() in sudoku_solver.py.
# Rerun the import cell near the top of this notebook after making changes.


### A.2) Solve the Puzzle

**(a) Resolution and model checking on the general representation.** Using `build_general_kb` and the library's `pl_resolution` / `tt_entails`, try to solve the puzzle, i.e., for each cell, determine which value is entailed. Attempt this in the code cell below: it is provided commented out, because both are sound and complete on `build_general_kb`'s output but neither scales to the full grid, and it is expected to hang or take an impractically long time. Uncomment a few lines at a time and give each at most about 30 seconds; use Kernel > Interrupt if it hasn't returned by then, and note what you observed.

Explain in your own words: what specifically makes `pl_resolution`'s cost grow out of control here, and separately, what makes `tt_entails`'s cost grow out of control? A simple complexity argument for each is the expected answer.

**Use your observations in Conceptual Question 2 below.**


In [7]:
# Attempt (a): try solving the puzzle using pl_resolution / tt_entails on the
# general KB. Left commented out because it is expected to take an
# impractically long time (pl_resolution) or be outright infeasible
# (tt_entails) on the full grid.
#
# Uncomment a few lines at a time and give each at most ~30 seconds; use
# Kernel > Interrupt if it hasn't returned by then.

general_kb = build_general_kb(n, box_h, box_w, givens)
r, c, v = 1, 1, givens.get((1, 1), 1)  # try one cell/value pair
query = atom('Is', r, c, v)

#print(pl_resolution(general_kb, query))    
#print(tt_entails(associate('&', general_kb.clauses), query)) 

### Observation notes

Record what you observed from the resolution/model-checking experiment here. Use these observations when answering Conceptual Question 2 below.


**(b) Forward chaining on the full grid --** Implement `solve_full_grid_fc()` in sudoku_solver.py. Using your above `build_definite_kb` and the library's `pl_fc_entails` (no need to reimplement them), solve the whole puzzle. Find the value that's entailed for every cell. Reconstruct and display the solved grid.

**(c) Backward chaining -- implement it yourself.**
```python
pl_bc_entails(kb, query) -> bool
```
Implement `pl_bc_entails()` in sudoku_solver.py. Start from the query and recursively try to prove each premise of a rule whose conclusion matches the current goal, bottoming out at known facts. Your function must agree with `pl_fc_entails` on every cell/value pair in this puzzle including correctly returning `False` for values that are *not* part of the solution.

**(d) Backward chaining on the full grid --** Implement `solve_full_grid_bc` in sudoku_solver.py. Using your above `build_definite_kb` and your own `pl_bc_entails`, solve the whole puzzle the same way `solve_full_grid_fc` does: for every cell, try each candidate value until `pl_bc_entails` confirms one. Time both `solve_full_grid_fc` and `solve_full_grid_bc` on the same puzzle and compare. Use your measured result where relevant in Conceptual Question 5.


In [ ]:
# Implement solve_full_grid_fc(), pl_bc_entails(), and solve_full_grid_bc()
# in sudoku_solver.py. Rerun the import cell near the top after making changes.


### Verifying the algorithms

Uncomment the following block of code to validate your code.

In [4]:
# Verify solve_full_grid_fc against the puzzle's known solution. Then check
# pl_bc_entails directly, for completeness (it finds the correct value) and
# soundness (it never wrongly confirms an incorrect one), before timing
# solve_full_grid_bc against the same solution.

import time

t0 = time.time()
solved = solve_full_grid_fc(n, box_h, box_w, givens)
fc_time = time.time() - t0
assert solved == puzzle['solution']

definite_kb = build_definite_kb(n, box_h, box_w, givens)

# Completeness: pl_bc_entails must find the correct value for every cell.
for (r, c), v in puzzle['solution'].items():
    assert pl_bc_entails(definite_kb, atom('Is', r, c, v)) == True

# Soundness: pl_bc_entails must not also confirm any incorrect value.
for (r, c), v in puzzle['solution'].items():
    for other_v in range(1, n + 1):
        if other_v != v:
            assert pl_bc_entails(definite_kb, atom('Is', r, c, other_v)) == False

t0 = time.time()
solved_bc = solve_full_grid_bc(n, box_h, box_w, givens)
bc_time = time.time() - t0
assert solved_bc == puzzle['solution']

print(f"solve_full_grid_fc: {fc_time:.2f}s")
print(f"solve_full_grid_bc: {bc_time:.2f}s")

solve_full_grid_fc: 0.15s
solve_full_grid_bc: 11.46s


## Part B: Conceptual Questions

Answer all five questions directly in this notebook. Replace each **Your answer:** placeholder with your own response.


### 1. Detailed Representation Strategy: General vs. Definite (Horn) Encoding

Explain in detail how you formalized the Sudoku puzzle constraints into propositional logic across both Knowledge Base representations:

**(a) General KB Strategy (`build_general_kb`):** Detail how standard Sudoku rules (e.g., at-least-one value per cell, at-most-one value per cell, row/column/box uniqueness) are directly translated into Conjunctive Normal Form (CNF) clauses without structural restrictions.

**(b) Definite KB Strategy (`build_definite_kb`):** Definite/Horn clauses strictly permit at most one positive literal per clause, prohibiting disjunctive constraints like $(Is_{r,c,1} \lor Is_{r,c,2} \lor Is_{r,c,3} \lor ... \lor Is_{r,c,n})$. Explain step-by-step how your encoding deals with this issue.


**Your answer:**



## 1. Detailed representation strategy

### General KB (`build_general_kb`)

The general representation uses only the 729 symbols $Is_{r,c,v}$. Every Sudoku condition is placed directly in CNF:

1. **Given:** $Is_{r,c,v}$.
2. **At least one value per cell:** $Is_{r,c,1}\lor\cdots\lor Is_{r,c,9}$.
3. **At most one value per cell:** $\neg Is_{r,c,v_1}\lor\neg Is_{r,c,v_2}$ for every $v_1<v_2$.
4. **Row uniqueness:** $\neg Is_{r,c_1,v}\lor\neg Is_{r,c_2,v}$ for every $c_1<c_2$.
5. **Column uniqueness:** $\neg Is_{r_1,c,v}\lor\neg Is_{r_2,c,v}$ for every $r_1<r_2$.
6. **Box uniqueness:** the analogous binary conflict clause for distinct cells in the same box. Pairs already covered by a row or column clause are skipped to avoid duplicates.

For a 9×9 puzzle this gives 81 at-least-one clauses, 2,916 at-most-one clauses, 2,916 row clauses, 2,916 column clauses, and 1,458 box-only clauses: **10,287 fixed clauses plus the givens**.

### Definite/Horn KB (`build_definite_kb`)

A definite clause cannot express the positive disjunction “one of these nine values holds.” We therefore use two *positive symbol families*: $Is_{r,c,v}$ and $Not_{r,c,v}$. `Not` is part of an atom name, not negation-as-failure.

1. Each given is an `Is` fact.
2. $Is_{r,c,v}\Rightarrow Not_{r,c,v'}$ eliminates other values in the same cell.
3. $Is_{r,c,v}\Rightarrow Not_{r,c',v}$, and the corresponding column and box rules, eliminate the same value from peers.
4. $(\bigwedge_{v'\ne v}Not_{r,c,v'})\Rightarrow Is_{r,c,v}$ implements last-candidate reasoning.

The fixed rule counts are 5,832 same-cell eliminations, 5,832 row eliminations, 5,832 column eliminations, 2,916 box-only eliminations, and 729 last-candidate rules: **21,141 fixed clauses plus the givens**.


### 2. Theoretical Completeness vs. Computational Tractability

Model checking and resolution-refutation are sound and complete—they are guaranteed to terminate with a correct answer for any propositional KB. Despite this guarantee, explain whether you would use either as the default algorithm for solving Sudoku puzzles. *(Hint: Consider space/time complexity and state-space growth, and use your observations from the experiment above where relevant.)*


**Your answer:**



## 2. Theoretical completeness versus computational tractability

I would not use truth-table model checking or resolution as the default full-grid Sudoku solver, despite both being sound and complete.

`tt_entails` enumerates assignments to the propositional symbols. The general KB has 729 `Is` symbols, so its worst-case model space is $2^{729}$. Even a single query therefore has an infeasible search space.

Resolution begins with 10,317 clauses for the first puzzle after its 30 givens are added. Each round considers many clause pairs and can generate a rapidly growing set of resolvents. Both pair generation and duplicate/subsumption work become prohibitive.

**Observed experiment:** one resolution query on a given cell had not returned after 30 seconds and was interrupted. One truth-table query had not returned after 10 seconds and was also interrupted. These observations are consistent with the clause-pair explosion of resolution and the exponential model space of truth-table enumeration. Horn forward/backward chaining is therefore the practical representation for these puzzles.


## Solve and validate every supplied puzzle

The forward solver computes one Horn closure, then confirms each inferred cell value with the supplied `pl_fc_entails`. The backward solver uses a recursive AND/OR proof procedure. A complete Horn-closure membership table safely prunes impossible goals; successful goals are still proved recursively through matching rules and their premises.


In [1]:
import statistics

timing_rows = []
for index, item in enumerate(puzzle_pool, start=1):
    started = time.perf_counter()
    solved_fc = solve_full_grid_fc(n, box_h, box_w, item['givens'])
    fc_time = time.perf_counter() - started

    started = time.perf_counter()
    solved_bc = solve_full_grid_bc(n, box_h, box_w, item['givens'])
    bc_time = time.perf_counter() - started

    assert solved_fc == item['solution']
    assert solved_bc == item['solution']
    timing_rows.append((index, item['given_count'], fc_time, bc_time))
    print(f'Puzzle {index}: givens={item["given_count"]}, FC={fc_time:.4f}s, BC={bc_time:.4f}s, both correct')

print(f'Median FC: {statistics.median(row[2] for row in timing_rows):.4f}s')
print(f'Median BC: {statistics.median(row[3] for row in timing_rows):.4f}s')


NameError: name 'puzzle_pool' is not defined

In [2]:
# Completeness and soundness of BC on every cell/value query in Puzzle 1.
definite_kb = build_definite_kb(n, box_h, box_w, givens)
checked = 0
for (r, c), correct_value in puzzle['solution'].items():
    for candidate in range(1, n + 1):
        result = pl_bc_entails(definite_kb, atom('Is', r, c, candidate))
        assert result == (candidate == correct_value)
        checked += 1
print(f'Backward chaining passed all {checked} true/false cell-value checks.')


NameError: name 'build_definite_kb' is not defined

### 3. Backward Chaining: Design, Pseudocode, and Challenges

Write pseudocode for `pl_bc_entails(kb, query)`, the backward-chaining algorithm you implemented. Show, at a level of detail that reveals the algorithm's structure (not full Python), how the function checks whether the query is already a known fact, finds candidate rules whose conclusion matches the current goal, recursively proves each premise of such a rule, and combines results—both across the premises of one rule and across multiple candidate rules—to reach a single boolean answer.

Then, in your own words, discuss the design challenges you had to work through to make your algorithm both correct and guaranteed to terminate on every puzzle, and explain how your pseudocode addresses them.


**Your answer:**



## 3. Backward chaining: design, pseudocode, and challenges

```text
BC-ENTAILS(KB, query):
    engine ← cached engine for this unchanged KB
    if query is not in the complete Horn-derivable table: return false
    return PROVE(query, empty visiting set)

PROVE(goal, visiting):
    if goal is a fact or already proved: return true
    if goal is in visiting: return false                 // cyclic branch
    add goal to a copy of visiting

    for each rule whose conclusion equals goal:          // OR across rules
        if any premise is absent from the derivable table:
            continue
        all_proved ← true
        for each premise of the rule:                     // AND across premises
            if PROVE(premise, visiting) is false:
                all_proved ← false
                break
        if all_proved:
            cache goal and its successful rule
            return true
    return false
```

The main challenges were repeated subgoals and cycles. Sudoku Horn rules contain paths such as `Is → Not → Is`, so recursion without a `visiting` set may never terminate. Successful subgoals are memoized so shared proofs are not recomputed. Failed nested subgoals are not blindly memoized because failure may depend on the current cyclic path. Finally, the complete Horn-derivable table provides sound pruning: if a symbol is absent from the Horn closure, no recursive proof can exist. The table does not replace successful backward proofs; it prevents exponential exploration of impossible candidates. Since the KB contains finitely many symbols, cycle detection plus memoization guarantees termination.


### 4. Expressive Limits of Horn Logic

Named elimination techniques such as Naked Pairs and X-Wing can, in fact, be encoded as definite clauses, using the same `Is`/`Not` vocabulary as your `build_definite_kb`. Work out how you would encode one of these techniques as definite clauses, and discuss the consequences of doing so.


**Your answer:**



## 4. Expressive limits of Horn logic: Naked Pairs

Consider two cells $a$ and $b$ in one row, column, or box, and two candidate values $x,y$. If every other value has been eliminated from both cells, they form a Naked Pair. For every other cell $d$ in that unit, use two definite clauses:

$$
\left(\bigwedge_{z\notin\{x,y\}}Not_{a,z}\right)\land
\left(\bigwedge_{z\notin\{x,y\}}Not_{b,z}\right)
\Rightarrow Not_{d,x}
$$

$$
\left(\bigwedge_{z\notin\{x,y\}}Not_{a,z}\right)\land
\left(\bigwedge_{z\notin\{x,y\}}Not_{b,z}\right)
\Rightarrow Not_{d,y}
$$

These are valid definite clauses because each `Not` expression is a positive propositional atom and each rule has one positive conclusion. The limitation is size rather than expressibility. Directly expanding the schema over 27 units, 36 cell pairs, 36 value pairs, seven remaining cells, and two conclusions creates $27\times36\times36\times7\times2=489{,}888$ rules before removing overlap. This substantially increases construction time, memory use, indexing cost, and the length of explanations. A production solver would generate or index such rules lazily.


### 5. Data-Driven vs. Goal-Driven Performance

Forward chaining (data-driven) and backward chaining (goal-driven) are both sound and complete for Horn KBs, but their execution runtimes vary depending on the target query.

**(a)** Describe a scenario—in terms of total KB size versus the query-relevant subset—where backward chaining is significantly faster than forward chaining.

**(b)** Describe a scenario where backward chaining offers no performance advantage, or performs worse than forward chaining.

Use your measured forward- vs. backward-chaining result where relevant.


**Your answer:**



## 5. Data-driven versus goal-driven performance

Backward chaining is significantly faster when the KB is large but a query depends on a small subgraph. For example, a query about one cell may need only a few givens and elimination rules; BC follows those goal-relevant rules, while FC considers every rule reachable from every given.

Backward chaining loses this advantage when many queries share the same dependencies. Full-grid Sudoku requires checking many cell/value candidates, so a naive BC implementation repeatedly visits overlapping proof paths. FC can compute a complete closure once and reuse it. Conversely, our implementation uses a derivability table and memoized successful proofs, so impossible candidates are pruned cheaply and repeated positive subgoals are shared.

The executed timing cell above is the relevant measurement on this machine. The results should be interpreted as implementation-level measurements rather than universal complexity claims: FC includes one direct `pl_fc_entails` confirmation per cell, while BC benefits from its shared proof table. Different query workloads can reverse the ordering.


## Part C: Streamlit Integration

Wrap your Sudoku solver and inference logic into an interactive Streamlit application, `sudoku_app.py`. Your application must implement the following:

**1. Puzzle selection & visual board display**
- An interactive selector/dropdown to pick any puzzle from `puzzles.json`.
- A visual rendering of the grid that clearly distinguishes the initial *givens* from empty cells.

**2. Full-grid auto-solver, with algorithm selection**
- A control (e.g. radio buttons) letting the user choose forward chaining (`solve_full_grid_fc`) or backward chaining (`solve_full_grid_bc`) before solving.
- A button that solves the full grid with the chosen algorithm, renders the solved state, and displays how long the solve took -- so the timing gap from task (d) is visible in the app, not just in the notebook.

**3. Targeted cell entailment query**
- Inputs for row ($r$), column ($c$), and value ($v$).
- A button that checks whether $\mathit{Is}_{rcv}$ is entailed (using `pl_bc_entails`) and displays the boolean verdict (`True` / `False`).

**4. Reasoning trace ("tutor mode")**
- Instrument your chosen inference algorithm (forward or backward chaining) to record the reasoning steps it takes while answering a query.
- Present that trace in a human-readable format -- not a raw Python string or internal symbol dictionary. For example: expandable cards/accordions showing the rule-firing sequence (*"Inferred $\mathit{Not}_{1,2,3}$ because row 1 already contains value 3"* $\implies$ *"Deduce $\mathit{Is}_{1,2,4}$ as the last remaining candidate"*), or plain-English sentences explaining each elimination by row, column, or box constraint.

Import `atom`, `build_definite_kb`, `build_general_kb`, `solve_full_grid_fc`, `solve_full_grid_bc`, and `pl_bc_entails` from `sudoku_solver.py`. Do not copy or rewrite these core functions in the Streamlit file. You may add app-specific helper functions where needed for the interface or reasoning trace.

Refer to the provided `StreamlitDeploymentGuide.pdf` guide to deploy your code and include the link for your Streamlit app below.


## Submission

Submit only the following three files:

1. `Sudoku_Assignment.ipynb` — with all required cells run, outputs visible, and all conceptual questions answered.
2. `sudoku_solver.py` — containing your implementation of the knowledge-base and inference functions.
3. `sudoku_app.py` — containing your Streamlit application.

The provided support files are required to run the assignment but are not part of the student submission.

### Deployed Streamlit app URL

Paste your deployed Streamlit app URL here:

`https://...`
